# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Viditp25/flyrank-intern-/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Baseline rule
The rule prioritizes pages for human review if they meet two conditions:
1. The page has not been updated for at least 91 days.
2. The page received at least 100 impressions during days 61–90, before the two comparison windows.

The score is the number of impressions in days 61–90 when both conditions are met. Pages that do not meet both conditions receive a score of zero and are excluded from the ranked queue.
- Reason code: STALE_BUT_VISIBLE
- Action label: REVIEW_FOR_REFRESH

The action means that a page should be reviewed by a person. It does not automatically mean the page should be rewritten.

The rule uses only days_since_last_update and impr_days_61_90. These inputs describe the page before the two comparison windows. The decline label and recent trend data are used only for evaluation and error analysis, never to calculate the score.
The 91-day threshold follows the existing freshness tier. The 100-impression threshold is the selected visibility floor. Neither threshold is tuned against the outcome.

In [1]:
# Section 1: Define the baseline rule
import numpy as np

STALE_DAYS = 91
VISIBLE_MIN = 100

SCORE_INPUTS = [
    "days_since_last_update",
    "impr_days_61_90",
]

REASON_CODE = "STALE_BUT_VISIBLE"
ACTION_LABEL = "REVIEW_FOR_REFRESH"


def baseline_rule(frame):
    """Score pages using only the declared pre-window inputs."""
    out = frame.copy()

    stale = (
        out["days_since_last_update"] >= STALE_DAYS
    ).astype(int)

    visible = (
        out["impr_days_61_90"] >= VISIBLE_MIN
    ).astype(int)

    out["score"] = (
        stale * visible * out["impr_days_61_90"]
    )

    out["reason_code"] = np.where(
        out["score"] > 0,
        REASON_CODE,
        ""
    )

    out["action_label"] = np.where(
        out["score"] > 0,
        ACTION_LABEL,
        ""
    )

    return out


print("Score inputs:", SCORE_INPUTS)
print("Reason code:", REASON_CODE)
print("Action label:", ACTION_LABEL)

Score inputs: ['days_since_last_update', 'impr_days_61_90']
Reason code: STALE_BUT_VISIBLE
Action label: REVIEW_FOR_REFRESH


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Ranked queue
The baseline rule is applied to every page in the dataset. Pages with positive scores are eligible for review and are sorted by score in descending order. If two pages have the same score, content_id provides a deterministic tie-break.
The ranked queue is saved to work/outputs/baseline_action_score.csv. The CSV contains the rank, pseudonymized content identifier, client identifier, score, reason code, action label, update age, and prior-period impressions.
The CSV is generated by the notebook and should remain excluded from Git according to the repository's data-safety rules.

In [2]:
# Section 2: Build the ranked queue and write the CSV

from pathlib import Path
import subprocess

import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Load and prepare the dataset if df is missing
# --------------------------------------------------

if "df" not in globals():
    DATA_REL = Path("data/raw/content_refresh_anonymized.csv")

    DATA_URL = (
        "https://raw.githubusercontent.com/"
        "Viditp25/flyrank-intern-/main/"
        "data/raw/content_refresh_anonymized.csv"
    )

    # Find the repository root if the repository is available
    def find_repo_root(start=None):
        start = Path(start or Path.cwd()).resolve()

        for path in [start, *start.parents]:
            if (path / DATA_REL).exists():
                return path

        return None

    REPO_ROOT = find_repo_root()

    # Load the dataset
    raw = pd.read_csv(
        REPO_ROOT / DATA_REL if REPO_ROOT else DATA_URL
    )

    df = raw.copy()

    # Derive impressions from days 61-90 only
    df["impr_days_61_90"] = (
        df["impressions_90d"]
        - df["impressions_last_30d"]
        - df["impressions_prev_30d"]
    )

    assert (df["impr_days_61_90"] >= 0).all(), (
        "Negative prior-period impressions detected."
    )

    # Evaluation label only; never used in scoring
    df["is_declining"] = (
        df["trend_direction"] == "down"
    ).astype(int)

    df["trend_defined"] = (
        df["impressions_prev_30d"] > 0
    )

    # Create short client codes for displayed results
    CLIENT_CODE = {
        cid: f"C{i + 1:02d}"
        for i, cid in enumerate(
            df["client_id"].value_counts().index
        )
    }

    df["client_code"] = df["client_id"].map(CLIENT_CODE)

    print("Dataset loaded:", df.shape)

else:
    print("Using the existing dataset in df:", df.shape)


# --------------------------------------------------
# 2. Define rule settings if not already defined
# --------------------------------------------------

STALE_DAYS = globals().get("STALE_DAYS", 91)
VISIBLE_MIN = globals().get("VISIBLE_MIN", 100)

REASON_CODE = globals().get(
    "REASON_CODE", "STALE_BUT_VISIBLE"
)

ACTION_LABEL = globals().get(
    "ACTION_LABEL", "REVIEW_FOR_REFRESH"
)

# --------------------------------------------------
# 3. Define the baseline rule if it is missing
# --------------------------------------------------

if "baseline_rule" not in globals():

    def baseline_rule(frame):
        out = frame.copy()

        stale = (
            out["days_since_last_update"] >= STALE_DAYS
        ).astype(int)

        visible = (
            out["impr_days_61_90"] >= VISIBLE_MIN
        ).astype(int)

        out["score"] = (
            stale
            * visible
            * out["impr_days_61_90"]
        )

        out["reason_code"] = np.where(
            out["score"] > 0,
            REASON_CODE,
            ""
        )

        out["action_label"] = np.where(
            out["score"] > 0,
            ACTION_LABEL,
            ""
        )

        return out


# --------------------------------------------------
# 4. Set the required output path
# --------------------------------------------------

if "OUT_PATH" not in globals():
    DATA_REL = Path("data/raw/content_refresh_anonymized.csv")

    REPO_ROOT = globals().get("REPO_ROOT")

    if REPO_ROOT is None:
        REPO_ROOT = find_repo_root()

    OUT_PATH = (
        (REPO_ROOT or Path.cwd())
        / "work"
        / "outputs"
        / "baseline_action_score.csv"
    )

OUT_REL = "work/outputs/baseline_action_score.csv"


# --------------------------------------------------
# 5. Score, filter, and rank eligible pages
# --------------------------------------------------

required_columns = [
    "content_id",
    "client_id",
    "days_since_last_update",
    "impr_days_61_90",
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

assert not missing_columns, (
    f"Required columns are missing: {missing_columns}"
)

scored = baseline_rule(df)

eligible = scored.loc[
    scored["score"] > 0
].copy()

ranked = (
    eligible
    .sort_values(
        ["score", "content_id"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

ranked["rank"] = ranked.index + 1


# --------------------------------------------------
# 6. Write the ranked queue
# --------------------------------------------------

CSV_COLUMNS = [
    "rank",
    "content_id",
    "client_id",
    "score",
    "reason_code",
    "action_label",
    "days_since_last_update",
    "impr_days_61_90",
]

OUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

ranked[CSV_COLUMNS].to_csv(
    OUT_PATH,
    index=False
)


# --------------------------------------------------
# 7. Display results and validate
# --------------------------------------------------

print(f"Total pages: {len(df):,}")
print(f"Eligible pages: {len(ranked):,}")
print(f"CSV saved to: {OUT_REL}")

display(
    ranked[
        [
            "rank",
            "content_id",
            "client_code",
            "score",
            "reason_code",
            "action_label",
        ]
    ].head(10)
)

assert OUT_PATH.exists(), "CSV was not created."

assert ranked["rank"].tolist() == list(
    range(1, len(ranked) + 1)
), "Rank sequence is invalid."

assert ranked["score"].is_monotonic_decreasing, (
    "Scores are not in descending order."
)

assert ranked["reason_code"].eq(
    REASON_CODE
).all(), "Unexpected reason code."

assert ranked["action_label"].eq(
    ACTION_LABEL
).all(), "Unexpected action label."

print("Ranked queue validation passed.")

Dataset loaded: (30000, 48)
Total pages: 30,000
Eligible pages: 7,506
CSV saved to: work/outputs/baseline_action_score.csv


,rank,content_id,client_code,score,reason_code,action_label
0,1,content_5fe46e04994d,C03,178138,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
1,2,content_2dba2b1f9536,C02,165634,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
2,3,content_cb112fce36be,C01,112942,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
3,4,content_b28d1efd668f,C02,84274,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
4,5,content_36ff89c8214e,C01,81700,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
5,6,content_2c2606c5d176,C01,79072,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
6,7,content_813e88069237,C02,76055,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
7,8,content_3d94572c3a35,C01,75252,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
8,9,content_124763d39ca5,C02,74850,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
9,10,content_396019fec61a,C03,71184,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH


Ranked queue validation passed.


1A. Signal audits (required before interpreting the score)
These checks use is_declining only as an evaluation outcome, not as a scoring input. A decline is measurable only when impressions_prev_30d > 0. Each table reports the number of measurable pages (n) and the observed decline rate by bucket.
Signal A — staleness / freshness flag. days_since_last_update is the measurable field corresponding to FlyRank's freshness/staleness concept. The rule uses the existing 91-day boundary. The audit tests whether older pages show a consistently higher decline rate; association alone does not prove a refresh will help.
Signal B — prior visibility. impr_days_61_90 is impressions from days 61–90, before both comparison windows. This checks whether more prior exposure predicts a higher decline rate. Visibility may still be useful as an exposure/materiality floor even if it is not a risk predictor.
Verdicts must be chosen from CONFIRMED, OPPOSITE, MIXED, or FALSE, based on the displayed bucket tables and client-level consistency. These are descriptive audits, not causal tests.

In [3]:
# Signal audits: descriptive buckets, sample counts, and explicit verdicts
# The outcome is used for evaluation only; it is not an input to baseline_rule.

import pandas as pd
import numpy as np

audit_df = df.loc[df["trend_defined"]].copy()
audit_df["is_declining"] = audit_df["is_declining"].astype(int)

# Signal A: update age / staleness
stale_bins = [-np.inf, 30, 90, 180, np.inf]
stale_labels = ["0-30 days", "31-90 days", "91-180 days", "181+ days"]
audit_df["staleness_bucket"] = pd.cut(
    audit_df["days_since_last_update"],
    bins=stale_bins,
    labels=stale_labels,
    right=True,
    include_lowest=True,
)

tbl_a = (
    audit_df.groupby("staleness_bucket", observed=False)
    .agg(n=("is_declining", "size"),
         n_declining=("is_declining", "sum"),
         decline_rate=("is_declining", "mean"))
    .reset_index()
)
tbl_a["decline_rate_pct"] = (100 * tbl_a["decline_rate"]).round(1)
print("SIGNAL A — STALENESS / FRESHNESS")
display(tbl_a[["staleness_bucket", "n", "n_declining", "decline_rate_pct"]])

# Client-level direction check: does 91+ day content have a higher decline rate
# than content updated within 30 days for each client where both groups exist?
audit_df["stale_91_plus"] = audit_df["days_since_last_update"] >= 91
client_stale = audit_df.groupby(["client_id", "stale_91_plus"])["is_declining"].mean().unstack()
if False in client_stale.columns and True in client_stale.columns:
    valid_stale = client_stale.dropna(subset=[False, True])
    stale_gap_pp = (valid_stale[True] - valid_stale[False]) * 100
    stale_clients_expected = int((stale_gap_pp > 0).sum())
    stale_clients_compared = int(len(stale_gap_pp))
    stale_weighted_gap_pp = float(
        np.average(stale_gap_pp, weights=audit_df.groupby("client_id").size().reindex(stale_gap_pp.index))
    ) if stale_clients_compared else np.nan
else:
    stale_gap_pp = pd.Series(dtype=float)
    stale_clients_expected = 0
    stale_clients_compared = 0
    stale_weighted_gap_pp = np.nan

# Signal B: impressions before the evaluation windows
visibility_bins = [-np.inf, 0, 99, 999, 9999, np.inf]
visibility_labels = ["0", "1-99", "100-999", "1,000-9,999", "10,000+"]
audit_df["visibility_bucket"] = pd.cut(
    audit_df["impr_days_61_90"],
    bins=visibility_bins,
    labels=visibility_labels,
    right=True,
    include_lowest=True,
)
tbl_b = (
    audit_df.groupby("visibility_bucket", observed=False)
    .agg(n=("is_declining", "size"),
         n_declining=("is_declining", "sum"),
         decline_rate=("is_declining", "mean"))
    .reset_index()
)
tbl_b["decline_rate_pct"] = (100 * tbl_b["decline_rate"]).round(1)
print("SIGNAL B — PRIOR VISIBILITY")
display(tbl_b[["visibility_bucket", "n", "n_declining", "decline_rate_pct"]])

# Client-level consistency for visibility: compare >=100 prior impressions
# with 1-99 impressions where both groups exist; zero-impression rows are separate.
audit_df["visibility_100_plus"] = audit_df["impr_days_61_90"] >= 100
client_vis = audit_df.groupby(["client_id", "visibility_100_plus"])["is_declining"].mean().unstack()
if False in client_vis.columns and True in client_vis.columns:
    valid_vis = client_vis.dropna(subset=[False, True])
    vis_gap_pp = (valid_vis[True] - valid_vis[False]) * 100
    vis_clients_expected = int((vis_gap_pp > 0).sum())
    vis_clients_compared = int(len(vis_gap_pp))
else:
    vis_gap_pp = pd.Series(dtype=float)
    vis_clients_expected = 0
    vis_clients_compared = 0

# Set these after reviewing the generated tables. The initial verdicts reflect
# the prior audit; change them if a fresh run produces materially different results.
VERDICTS = {
    "A_staleness (days_since_last_update; freshness/staleness concept)": "MIXED",
    "B_prior_visibility (impr_days_61_90)": "FALSE",
}
ALLOWED_VERDICTS = {"CONFIRMED", "OPPOSITE", "MIXED", "FALSE"}
assert set(VERDICTS.values()) <= ALLOWED_VERDICTS

print("Client consistency — staleness: "
      f"{stale_clients_expected}/{stale_clients_compared} clients show higher decline among 91+ day pages; "
      f"weighted gap={stale_weighted_gap_pp:.1f} pp" if stale_clients_compared else
      "Client consistency — staleness: not enough clients have both comparison groups.")
print("Client consistency — visibility: "
      f"{vis_clients_expected}/{vis_clients_compared} clients show higher decline among pages with 100+ prior impressions; "
      "comparison group is 1-99 impressions." if vis_clients_compared else
      "Client consistency — visibility: not enough clients have both comparison groups.")
display(pd.Series(VERDICTS, name="verdict").to_frame())

assert {"n", "n_declining", "decline_rate"} <= set(tbl_a.columns)
assert {"n", "n_declining", "decline_rate"} <= set(tbl_b.columns)
assert int(tbl_a["n"].sum()) == len(audit_df)
assert int(tbl_b["n"].sum()) == len(audit_df)


SIGNAL A — STALENESS / FRESHNESS


,staleness_bucket,n,n_declining,decline_rate_pct
0,0-30 days,17454,10473,60.0
1,31-90 days,167,103,61.7
2,91-180 days,8858,5604,63.3
3,181+ days,133,82,61.7


SIGNAL B — PRIOR VISIBILITY


,visibility_bucket,n,n_declining,decline_rate_pct
0,0,959,546,56.9
1,1-99,6426,3931,61.2
2,100-999,9782,5983,61.2
3,"1,000-9,999",8110,5034,62.1
4,"10,000+",1335,768,57.5


Client consistency — staleness: 12/21 clients show higher decline among 91+ day pages; weighted gap=4.8 pp
Client consistency — visibility: 12/28 clients show higher decline among pages with 100+ prior impressions; comparison group is 1-99 impressions.


,verdict
A_staleness (days_since_last_update; freshness/staleness concept),MIXED
B_prior_visibility (impr_days_61_90),FALSE


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review
The top 20 pages are selected directly from the ranked queue. Each row is reviewed using its action, reason code, score, and observable characteristics.
The confidence note describes limitations in the available evidence. It is not a calibrated probability that the recommendation is correct.
The review also identifies missing position data, weak demand evidence, unusually deep search positions, and update dates shared by many pages from the same client. These checks help identify cases where the rule may prioritize a page incorrectly.
The review-only fields are not used to modify the score or reorder the queue.

In [4]:
# Section 3: Review the top 20 ranked pages

import pandas as pd
import numpy as np

# Select the top 20 actual ranked rows
top20 = ranked.head(20).copy()

assert len(top20) == min(20, len(ranked)), (
    "Unexpected number of ranked rows."
)

# Check optional review columns
optional_columns = [
    "avg_position",
    "ctr",
    "search_volume",
    "trend_direction",
]

for col in optional_columns:
    if col not in top20.columns:
        top20[col] = np.nan

# Get each client's most common update age
if "same_age_share" not in globals():
    same_age_share = (
        ranked.groupby("client_id")["days_since_last_update"]
        .agg(
            lambda s: (
                s.value_counts().iloc[0] / len(s)
                if len(s) > 0 else 0
            )
        )
    )


# Explain why a page ranked highly
def explain_rank(r):
    return (
        f"Last updated {int(r['days_since_last_update'])} days ago; "
        f"prior-period impressions = {int(r['impr_days_61_90']):,}. "
        f"The rule requires update age >= {STALE_DAYS} days "
        f"and prior impressions >= {VISIBLE_MIN}. "
        f"The score equals prior-period impressions."
    )


# Identify possible reasons the recommendation could be wrong
def explain_risk(r):
    concerns = []

    if pd.isna(r["avg_position"]):
        concerns.append("average position unavailable")
    elif r["avg_position"] == 0:
        concerns.append("no position data")
    elif r["avg_position"] > 20:
        concerns.append("average position is beyond page two")

    if (
        pd.isna(r["search_volume"])
        or r["search_volume"] == 0
    ):
        concerns.append("keyword search volume missing or zero")

    client_id = r["client_id"]
    if same_age_share.get(client_id, 0) >= 0.9:
        concerns.append(
            "update age may reflect a client-wide batch or sync"
        )

    if r["trend_direction"] in ("stable", "up"):
        concerns.append(
            "observed trend is stable or upward, not declining"
        )

    if (
        pd.notna(r["impressions_last_30d"])
        and r["impr_days_61_90"] > 0
    ):
        ratio = (
            r["impressions_last_30d"]
            / r["impr_days_61_90"]
        )

        if ratio < 0.25:
            concerns.append(
                "recent impressions are below 25% of prior-period impressions"
            )

    concerns.append(
        "seasonality, search intent, actual edit history, and refresh impact are unknown"
    )

    return "; ".join(concerns)


# Assign the review fields
top20["why_ranked_highly"] = top20.apply(
    explain_rank,
    axis=1
)

top20["could_be_wrong_because"] = top20.apply(
    explain_risk,
    axis=1
)


# Add a cautious confidence note
def confidence_note(r):
    if (
        pd.isna(r["avg_position"])
        or r["avg_position"] == 0
        or pd.isna(r["search_volume"])
        or r["search_volume"] == 0
        or r["avg_position"] > 20
        or same_age_share.get(r["client_id"], 0) >= 0.9
        or r["trend_direction"] in ("stable", "up")
    ):
        return (
            "LOWER: one or more evidence limitations require human review"
        )

    return (
        "PROVISIONAL: passes the rule's gates; "
        "recommendation still requires human review"
    )


top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)


# Display the complete review
print(f"TOP-20 REVIEW — {len(top20)} ranked items")

display(
    top20[
        [
            "rank",
            "content_id",
            "client_code",
            "action_label",
            "reason_code",
            "score",
            "confidence_note",
            "why_ranked_highly",
            "could_be_wrong_because",
        ]
    ]
)


# Validate the review
required_review_columns = [
    "action_label",
    "reason_code",
    "confidence_note",
    "why_ranked_highly",
    "could_be_wrong_because",
]

for col in required_review_columns:
    assert top20[col].notna().all(), (
        f"Missing values found in {col}"
    )
    assert top20[col].astype(str).str.strip().ne("").all(), (
        f"Empty review entries found in {col}"
    )

print("Top-20 review validation passed.")

TOP-20 REVIEW — 20 ranked items


,rank,content_id,client_code,action_label,reason_code,score,confidence_note,why_ranked_highly,could_be_wrong_because
0,1,content_5fe46e04994d,C03,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,178138,LOWER: one or more evidence limitations requir...,Last updated 104 days ago; prior-period impres...,update age may reflect a client-wide batch or ...
1,2,content_2dba2b1f9536,C02,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,165634,LOWER: one or more evidence limitations requir...,Last updated 104 days ago; prior-period impres...,average position is beyond page two; keyword s...
2,3,content_cb112fce36be,C01,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,112942,LOWER: one or more evidence limitations requir...,Last updated 104 days ago; prior-period impres...,update age may reflect a client-wide batch or ...
3,4,content_b28d1efd668f,C02,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,84274,LOWER: one or more evidence limitations requir...,Last updated 104 days ago; prior-period impres...,average position is beyond page two; keyword s...
4,5,content_36ff89c8214e,C01,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,81700,LOWER: one or more evidence limitations requir...,Last updated 104 days ago; prior-period impres...,keyword search volume missing or zero; update ...
5,6,content_2c2606c5d176,C01,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,79072,LOWER: one or more evidence limitations requir...,Last updated 104 days ago; prior-period impres...,update age may reflect a client-wide batch or ...
6,7,content_813e88069237,C02,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,76055,LOWER: one or more evidence limitations requir...,Last updated 104 days ago; prior-period impres...,average position is beyond page two; keyword s...
7,8,content_3d94572c3a35,C01,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,75252,LOWER: one or more evidence limitations requir...,Last updated 104 days ago; prior-period impres...,update age may reflect a client-wide batch or ...
8,9,content_124763d39ca5,C02,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,74850,LOWER: one or more evidence limitations requir...,Last updated 104 days ago; prior-period impres...,average position is beyond page two; keyword s...
9,10,content_396019fec61a,C03,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,71184,LOWER: one or more evidence limitations requir...,Last updated 104 days ago; prior-period impres...,update age may reflect a client-wide batch or ...


Top-20 review validation passed.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks and leakage check
A weak pick is a page that meets the baseline rule but may not be a useful refresh opportunity. For example, its recent impressions may have collapsed, it may not currently be declining, or it may have poor search-position and demand evidence.
The weak-pick checks are used only to review the ranked recommendations after scoring. They do not change the score.
The leakage check verifies that the rule uses only days_since_last_update and impr_days_61_90. The outcome label, the two comparison windows, and existing product flags must not be predictors.
This baseline is intended for decision support. Its ranking does not prove that refreshing a page will improve performance.

In [5]:

# Section 4: Weak picks + leakage check

import pandas as pd
import numpy as np

# ==================================================
# PART A: WEAK-PICK ANALYSIS
# ==================================================

# Review the top 100 ranked pages without changing their scores.
weak_picks = ranked.head(100).copy()

# Map additional evidence from the original dataset.
# Assumes each content_id identifies one page.
source_df = (
    df.drop_duplicates(subset="content_id")
      .set_index("content_id")
)

optional_columns = [
    "avg_position",
    "ctr",
    "search_volume",
    "trend_direction",
    "impressions_last_30d",
    "impressions_prev_30d",
]

for col in optional_columns:
    if col in source_df.columns and col not in weak_picks.columns:
        weak_picks[col] = weak_picks["content_id"].map(source_df[col])

# Store reasons why a recommendation may need extra review.
weak_picks["weak_pick_reasons"] = ""

# 1. Check average search position.
if "avg_position" in weak_picks.columns:
    missing_position = (
        weak_picks["avg_position"].isna()
        | weak_picks["avg_position"].eq(0)
    )

    weak_picks.loc[
        missing_position, "weak_pick_reasons"
    ] += "Missing or zero average position; "

    weak_picks.loc[
        weak_picks["avg_position"].gt(20),
        "weak_pick_reasons"
    ] += "Average position beyond top 20; "

# 2. Check search-volume evidence.
if "search_volume" in weak_picks.columns:
    missing_volume = (
        weak_picks["search_volume"].isna()
        | weak_picks["search_volume"].eq(0)
    )

    weak_picks.loc[
        missing_volume, "weak_pick_reasons"
    ] += "Missing or zero search volume; "

# 3. Check observed trend.
if "trend_direction" in weak_picks.columns:
    trend = (
        weak_picks["trend_direction"]
        .astype("string")
        .str.lower()
        .str.strip()
    )

    weak_picks.loc[
        trend.isin(["stable", "up", "upward"]),
        "weak_pick_reasons"
    ] += "Trend is stable or upward; "

# 4. Compare recent impressions with days 61-90.
if (
    "impressions_last_30d" in weak_picks.columns
    and "impr_days_61_90" in weak_picks.columns
):
    recent_impressions = weak_picks["impressions_last_30d"]
    older_impressions = weak_picks["impr_days_61_90"]

    weak_picks.loc[
        recent_impressions < 0.25 * older_impressions,
        "weak_pick_reasons"
    ] += "Recent impressions below 25% of days 61-90 impressions; "

# Clean up reason text and flag pages with concerns.
weak_picks["weak_pick_reasons"] = (
    weak_picks["weak_pick_reasons"]
    .str.strip()
    .str.rstrip(";")
)

weak_picks["flagged_as_weak"] = (
    weak_picks["weak_pick_reasons"].ne("")
)

print("WEAK-PICK ANALYSIS")
print("------------------")
print("Ranked pages reviewed:", len(weak_picks))
print(
    "Pages flagged for evidence concerns:",
    int(weak_picks["flagged_as_weak"].sum())
)

display_columns = [
    col for col in [
        "rank",
        "content_id",
        "client_id",
        "score",
        "avg_position",
        "search_volume",
        "trend_direction",
        "weak_pick_reasons",
    ]
    if col in weak_picks.columns
]

display(
    weak_picks.loc[
        weak_picks["flagged_as_weak"],
        display_columns
    ].head(20)
)

# ==================================================
# PART B: LEAKAGE CHECK
# ==================================================

# Evaluation labels, comparison-window variables, future outcomes,
# and known product/page flags must not be score inputs.
#
# Check df.columns below and add any other actual product-flag
# column names present in your dataset.

BANNED = [
    # Outcome and evaluation fields
    "trend_direction",
    "is_declining",
    "trend_defined",
    "decline_rate",

    # Comparison windows and future outcomes
    "impressions_last_30d",
    "impressions_prev_30d",
    "future_performance",
    "future_clicks",
    "future_impressions",
    "future_outcome",

    # Common product/page flag names
    "is_product",
    "is_product_page",
    "product_flag",
]

assert "SCORE_INPUTS" in globals(), (
    "SCORE_INPUTS is missing. Run Section 1 first."
)

leaked_inputs = set(SCORE_INPUTS).intersection(BANNED)

print("\nLEAKAGE CHECK")
print("-------------")
print("Declared score inputs:", SCORE_INPUTS)
print("Forbidden inputs:", BANNED)

assert not leaked_inputs, (
    f"Leakage detected in SCORE_INPUTS: {sorted(leaked_inputs)}"
)

required_score_inputs = {
    "days_since_last_update",
    "impr_days_61_90",
}

assert set(SCORE_INPUTS) == required_score_inputs, (
    "Scoring inputs differ from the declared baseline rule."
)

print("PASS: Declared inputs contain no listed forbidden fields.")
print("PASS: Scoring inputs match the baseline rule.")

# ==================================================
# PART C: VALIDATE THE SCORING FORMULA
# ==================================================

# Recalculate the expected score using the declared rule.
expected_score = (
    (ranked["days_since_last_update"] >= STALE_DAYS).astype(int)
    * (ranked["impr_days_61_90"] >= VISIBLE_MIN).astype(int)
    * ranked["impr_days_61_90"]
)

assert np.allclose(
    ranked["score"].to_numpy(dtype=float),
    expected_score.to_numpy(dtype=float),
    equal_nan=False
), "Score does not match the declared baseline formula."

assert ranked["score"].gt(0).all(), (
    "The ranked queue contains a non-positive score."
)

assert ranked["reason_code"].eq(REASON_CODE).all(), (
    "Unexpected reason code found in ranked queue."
)

assert ranked["action_label"].eq(ACTION_LABEL).all(), (
    "Unexpected action label found in ranked queue."
)

# Verify descending scores and deterministic tie-breaking.
assert ranked["score"].is_monotonic_decreasing, (
    "Ranked scores are not in descending order."
)

expected_order = (
    ranked.sort_values(
        ["score", "content_id"],
        ascending=[False, True]
    )["content_id"]
    .reset_index(drop=True)
)

assert ranked["content_id"].reset_index(drop=True).equals(
    expected_order
), "Ranking or content_id tie-breaking is incorrect."

print("PASS: Scores match the declared baseline formula.")
print("PASS: All ranked rows have the expected reason code and action.")
print("PASS: Scores are descending with deterministic tie-breaking.")
print("PASS: Weak-pick flags do not modify scores or ranking.")


WEAK-PICK ANALYSIS
------------------
Ranked pages reviewed: 100
Pages flagged for evidence concerns: 79


,rank,content_id,client_id,score,avg_position,search_volume,trend_direction,weak_pick_reasons
1,2,content_2dba2b1f9536,client_6208ef0f77,165634,27.9,0.0,stable,Average position beyond top 20; Missing or zer...
3,4,content_b28d1efd668f,client_6208ef0f77,84274,26.2,0.0,stable,Average position beyond top 20; Missing or zer...
4,5,content_36ff89c8214e,client_19581e27de,81700,7.3,0.0,stable,Missing or zero search volume; Trend is stable...
6,7,content_813e88069237,client_6208ef0f77,76055,26.2,0.0,down,Average position beyond top 20; Missing or zer...
8,9,content_124763d39ca5,client_6208ef0f77,74850,33.2,0.0,down,Average position beyond top 20; Missing or zer...
9,10,content_396019fec61a,client_4e07408562,71184,3.8,10.0,stable,Trend is stable or upward; Recent impressions ...
10,11,content_c5063073d048,client_6208ef0f77,70433,12.5,0.0,stable,Missing or zero search volume; Trend is stable...
11,12,content_05e9b4cd9ccf,client_6208ef0f77,69614,22.1,0.0,down,Average position beyond top 20; Missing or zer...
13,14,content_8b36799b7e44,client_6208ef0f77,68432,32.0,0.0,down,Average position beyond top 20; Missing or zer...
16,17,content_3430a8b94511,client_19581e27de,60888,3.3,10.0,stable,Trend is stable or upward



LEAKAGE CHECK
-------------
Declared score inputs: ['days_since_last_update', 'impr_days_61_90']
Forbidden inputs: ['trend_direction', 'is_declining', 'trend_defined', 'decline_rate', 'impressions_last_30d', 'impressions_prev_30d', 'future_performance', 'future_clicks', 'future_impressions', 'future_outcome', 'is_product', 'is_product_page', 'product_flag']
PASS: Declared inputs contain no listed forbidden fields.
PASS: Scoring inputs match the baseline rule.
PASS: Scores match the declared baseline formula.
PASS: All ranked rows have the expected reason code and action.
PASS: Scores are descending with deterministic tie-breaking.
PASS: Weak-pick flags do not modify scores or ranking.


In [6]:
# Self-check: Top-20 review requirements

import pandas as pd

# Define the check helper
checks = []

def check(name, ok, detail=""):
    checks.append({
        "requirement": name,
        "pass": bool(ok),
        "detail": detail
    })


# 1. Top-20 review contains actual ranked rows
check(
    "Top-20 review contains up to 20 actual ranked rows",
    len(top20) == min(20, len(ranked)),
    f"Reviewed {len(top20)} rows"
)

# 2. Each row has an action and reason code
check(
    "Each reviewed row has an action and reason code",
    top20["action_label"].notna().all()
    and top20["action_label"].astype(str).str.strip().ne("").all()
    and top20["reason_code"].notna().all()
    and top20["reason_code"].astype(str).str.strip().ne("").all()
)

# 3. Each row has a confidence note
check(
    "Each reviewed row has a confidence note",
    top20["confidence_note"].notna().all()
    and top20["confidence_note"].astype(str).str.strip().ne("").all()
)

# 4. Each row explains why it ranked highly
check(
    "Each reviewed row explains why it ranked highly",
    top20["why_ranked_highly"].notna().all()
    and top20["why_ranked_highly"].astype(str).str.strip().ne("").all()
)

# 5. Each row explains what could make it wrong
check(
    "Each reviewed row explains what could make it wrong",
    top20["could_be_wrong_because"].notna().all()
    and top20["could_be_wrong_because"].astype(str).str.strip().ne("").all()
)

# 6. CSV exists at the required path
check(
    "CSV exists at the required path",
    OUT_PATH.exists(),
    str(OUT_REL)
)

# 7. Scoring uses only declared inputs
check(
    "Scoring uses only the declared inputs",
    not set(SCORE_INPUTS).intersection(BANNED),
    str(SCORE_INPUTS)
)

# 8. CSV has all expected columns and ranked rows
csv_back = pd.read_csv(OUT_PATH) if OUT_PATH.exists() else pd.DataFrame()

check(
    "CSV contains the expected columns",
    not csv_back.empty
    and set(CSV_COLUMNS).issubset(csv_back.columns),
    f"CSV rows: {len(csv_back)}"
)

check(
    "CSV ranks are sequential and scores are descending",
    not csv_back.empty
    and csv_back["rank"].tolist() == list(range(1, len(csv_back) + 1))
    and csv_back["score"].is_monotonic_decreasing
)

# 9. Two signal audits include bucket tables and sample counts
check(
    "Signal A staleness audit has bucket counts",
    {"n", "n_declining", "decline_rate"} <= set(tbl_a.columns)
    and int(tbl_a["n"].sum()) == len(audit_df),
    f"Signal A audited rows: {int(tbl_a['n'].sum()):,}"
)
check(
    "Signal B prior-visibility audit has bucket counts",
    {"n", "n_declining", "decline_rate"} <= set(tbl_b.columns)
    and int(tbl_b["n"].sum()) == len(audit_df),
    f"Signal B audited rows: {int(tbl_b['n'].sum()):,}"
)
check(
    "Both signal verdicts use the allowed vocabulary",
    len(VERDICTS) == 2
    and set(VERDICTS.values()) <= {"CONFIRMED", "OPPOSITE", "MIXED", "FALSE"},
    str(VERDICTS)
)
check(
    "Staleness audit is explicitly linked to freshness/staleness concept",
    any("staleness" in key.lower() for key in VERDICTS)
)

# Display results
result = pd.DataFrame(checks)
display(result)

# Fail visibly if any check fails
if not result["pass"].all():
    display(result.loc[~result["pass"]])
    raise AssertionError(
        "A self-check failed. Fix the failed checks before submitting."
    )

print("All code-checkable requirements pass.")

,requirement,pass,detail
0,Top-20 review contains up to 20 actual ranked ...,True,Reviewed 20 rows
1,Each reviewed row has an action and reason code,True,
2,Each reviewed row has a confidence note,True,
3,Each reviewed row explains why it ranked highly,True,
4,Each reviewed row explains what could make it ...,True,
5,CSV exists at the required path,True,work/outputs/baseline_action_score.csv
6,Scoring uses only the declared inputs,True,"['days_since_last_update', 'impr_days_61_90']"
7,CSV contains the expected columns,True,CSV rows: 7506
8,CSV ranks are sequential and scores are descen...,True,
9,Signal A staleness audit has bucket counts,True,"Signal A audited rows: 26,612"


All code-checkable requirements pass.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.